# Base imports

In [1]:
library(devtools)

Loading required package: usethis



In [2]:
library(tidyverse)

── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
library(here)

here() starts at /n/scratch/users/r/rob6090/projects/tdp-43/scripts/analysis/final_dsRNA



In [4]:
library(powerjoin)

In [5]:
library(qs)

qs 0.27.3. Announcement: https://github.com/qsbase/qs/issues/103



In [6]:
library(Rsubread)

In [7]:
library(ggbeeswarm)

In [8]:
library(Rsubread)

In [9]:
library(GenomicRanges)

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:lubridate’:

    intersect, setdiff, union


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:lubridate’:

    second, second<-


The

In [10]:
# Prepare BAM files
bam_files <- tibble(
  bam_path = Sys.glob(file.path("/home/rob6090/scr_proj/tdp-43/data/samples_bam/*.sorted.bam"))
) %>%
  filter(!str_detect(bam_path, fixed("markdup"))) %>%
  mutate(
    sample_id = str_extract(bam_path, "([^/]+)\\.sorted\\.bam", group = 1)
  )

In [ ]:
bam_files

In [12]:
# Load RepeatMasker file
repeatmasker_raw <- read_csv("~/main/projects/genomics/tdp-43/data/repeatmasker_raw.csv") # Assume file named repeatmasker_raw.csv

Rows: 5448004 Columns: 16
── Column specification ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (8): chromosome, left, strand, rep_name, class_family, rep_start, rep_le...
dbl (8): sw_score, perc_div, perc_del, perc_ins, start, end, rep_end, id

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [13]:
repeats_saf <- repeatmasker_raw%>%
  transmute(
    GeneID = paste(rep_name, row_number(), sep = "_"),
    Chr = chromosome,
    Start = start,
    End = end,
    Strand = case_when(
      strand == "+" ~ "+",
      strand == "C" ~ "-",
      TRUE ~ "*"
    )
  )

In [14]:
1

[1] 1

In [15]:
head(repeats_saf)

GeneID,Chr,Start,End,Strand
<chr>,<chr>,<dbl>,<dbl>,<chr>
(TAACCC)n_1,chr1,10001,10468,+
TAR1_2,chr1,10469,11447,-
L1MC_3,chr1,11505,11675,-
MER5B_4,chr1,11678,11780,-
MIR3_5,chr1,15265,15355,-
(TGCTCC)n_6,chr1,15798,15849,+


In [16]:
# Run featureCounts
fc_results <- featureCounts(
  files = bam_files$bam_path,
  annot.ext = repeats_saf,
  isGTFAnnotationFile = FALSE,
  allowMultiOverlap = TRUE,
  countMultiMappingReads = TRUE,               # -M
  fraction = TRUE,               # --fraction
  primaryOnly = FALSE,
  isPairedEnd = TRUE,
  nthreads = 14
)


        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
       Rsubread 2.16.1

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 14 BAM files                                     ||
||                                                                            ||
||                           SRR8571937.sorted.bam                            ||
||                           SRR8571938.sorted.bam                            ||
||                           SRR8571939.sorted.bam  

## Unique only feature counts

In [19]:
# Run featureCounts
fc_results <- featureCounts(
  files = bam_files$bam_path,
  annot.ext = repeats_saf,
  isGTFAnnotationFile = FALSE,
  allowMultiOverlap = TRUE,
  #countMultiMappingReads = TRUE,               # -M
  #fraction = TRUE,               # --fraction
  primaryOnly = FALSE,
  isPairedEnd = TRUE,
  nthreads = 14
)


        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
       Rsubread 2.16.1

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 14 BAM files                                     ||
||                                                                            ||
||                           SRR8571937.sorted.bam                            ||
||                           SRR8571938.sorted.bam                            ||
||                           SRR8571939.sorted.bam  

In [21]:
2

[1] 2

In [16]:
qsave(
  fc_results,
  "/n/scratch/users/r/rob6090/projects/tdp-43/scripts/analysis/final_dsRNA/liu_repeat_counts_raw-fraction_AND_secondary_noMETA.qs"
)

# additional tests - fractional counts

In [18]:
type(fc_results)

[1] "list"

In [20]:
length(fc_results[1])

[1] 1

In [24]:
2

[1] 2

In [22]:
# Load the DESeq2 library
library(DESeq2)

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘matrixStats’


The following object is masked from ‘package:dplyr’:

    count



Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, 

In [25]:
# Extract the count matrix from your featureCounts results
count_matrix <- fc_results$counts

In [26]:
head(count_matrix)

,SRR8571937.sorted.bam,SRR8571938.sorted.bam,SRR8571939.sorted.bam,SRR8571940.sorted.bam,SRR8571941.sorted.bam,SRR8571942.sorted.bam,SRR8571944.sorted.bam,SRR8571945.sorted.bam,SRR8571947.sorted.bam,SRR8571948.sorted.bam,SRR8571949.sorted.bam,SRR8571950.sorted.bam,SRR8571951.sorted.bam,SRR8571952.sorted.bam
(TAACCC)n_1,0.00,0.67,0.00,0.08,0.00,0.00,0.50,0.00,0.00,0.00,2.00,0.00,0.36,0.00
TAR1_2,0.00,0.00,0.00,0.00,0.83,0.83,0.67,1.50,1.50,2.25,0.00,0.00,0.00,0.83
L1MC_3,0.77,0.38,0.25,0.25,0.17,0.00,0.00,0.60,0.79,2.31,1.17,1.35,0.50,1.29
MER5B_4,1.42,0.38,0.38,0.00,0.31,0.00,0.00,1.60,0.93,1.60,1.20,1.02,0.00,0.12
MIR3_5,0.40,2.24,3.25,1.71,0.62,0.50,5.43,0.95,1.12,1.32,3.25,2.41,0.00,0.00
(TGCTCC)n_6,3.53,4.84,9.32,9.05,4.15,6.94,5.49,15.21,9.64,8.38,11.40,3.50,2.83,0.60


In [27]:
colnames(count_matrix)

[1] "SRR8571937.sorted.bam" "SRR8571938.sorted.bam" "SRR8571939.sorted.bam"
 [4] "SRR8571940.sorted.bam" "SRR8571941.sorted.bam" "SRR8571942.sorted.bam"
 [7] "SRR8571944.sorted.bam" "SRR8571945.sorted.bam" "SRR8571947.sorted.bam"
[10] "SRR8571948.sorted.bam" "SRR8571949.sorted.bam" "SRR8571950.sorted.bam"
[13] "SRR8571951.sorted.bam" "SRR8571952.sorted.bam"

In [28]:
sample_info <- data.frame(
  sample_name = colnames(count_matrix),
  condition = factor(c("TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative")) # Adjust this to your actual sample conditions
)

In [29]:
sample_info

sample_name,condition
<chr>,<fct>
SRR8571937.sorted.bam,TDP43_positive
SRR8571938.sorted.bam,TDP43_negative
SRR8571939.sorted.bam,TDP43_positive
SRR8571940.sorted.bam,TDP43_negative
SRR8571941.sorted.bam,TDP43_positive
SRR8571942.sorted.bam,TDP43_negative
SRR8571944.sorted.bam,TDP43_positive
SRR8571945.sorted.bam,TDP43_negative
SRR8571947.sorted.bam,TDP43_positive


In [30]:
rownames(sample_info) <- colnames(count_matrix)

In [31]:
sample_info

,sample_name,condition
,<chr>,<fct>
SRR8571937.sorted.bam,SRR8571937.sorted.bam,TDP43_positive
SRR8571938.sorted.bam,SRR8571938.sorted.bam,TDP43_negative
SRR8571939.sorted.bam,SRR8571939.sorted.bam,TDP43_positive
SRR8571940.sorted.bam,SRR8571940.sorted.bam,TDP43_negative
SRR8571941.sorted.bam,SRR8571941.sorted.bam,TDP43_positive
SRR8571942.sorted.bam,SRR8571942.sorted.bam,TDP43_negative
SRR8571944.sorted.bam,SRR8571944.sorted.bam,TDP43_positive
SRR8571945.sorted.bam,SRR8571945.sorted.bam,TDP43_negative
SRR8571947.sorted.bam,SRR8571947.sorted.bam,TDP43_positive


In [33]:
1

[1] 1

In [34]:
# --- FIX: Round the fractional counts to the nearest integer ---
count_matrix <- round(count_matrix)

## Run Differential Expression Analysis

In [35]:
# Create the DESeqDataSet object
dds <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = sample_info,
  design = ~ condition
)

converting counts to integer mode



In [36]:
# Run the DESeq2 analysis
dds <- DESeq(dds)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

-- replacing outliers and refitting for 61872 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)

estimating dispersions

fitting model and testing



In [37]:
qsave(
  dds,
  file.path("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam", "data", "tdp-43_path_de.qs")
)

In [38]:
# Get the results
# The contrast specifies that we are comparing the 'TDP43_negative' group to the 'TDP43_positive' group.
res <- results(dds, contrast = c("condition", "TDP43_negative", "TDP43_positive"))

In [ ]:
res2 <- results(dds, contrast = c("condition",  "TDP43_positive", "TDP43_negative"))

In [39]:
type(res)

[1] "double"

In [40]:
head(res)

log2 fold change (MLE): condition TDP43_negative vs TDP43_positive 
Wald test p-value: condition TDP43_negative vs TDP43_positive 
DataFrame with 6 rows and 6 columns
             baseMean log2FoldChange     lfcSE      stat    pvalue      padj
            <numeric>      <numeric> <numeric> <numeric> <numeric> <numeric>
(TAACCC)n_1  0.195381     -0.4404094  3.084575 -0.142778  0.886466        NA
TAR1_2       0.699543      0.5526920  1.549296  0.356738  0.721288        NA
L1MC_3       0.535007      0.6424273  1.630292  0.394057  0.693539        NA
MER5B_4      0.518370      0.5945308  1.762943  0.337238  0.735938        NA
MIR3_5       1.380022     -0.7457321  1.081602 -0.689470  0.490527        NA
(TGCTCC)n_6  6.346198      0.0845385  0.546973  0.154557  0.877170  0.994814

In [45]:
head(res,n=10)

log2 fold change (MLE): condition TDP43_negative vs TDP43_positive 
Wald test p-value: condition TDP43_negative vs TDP43_positive 
DataFrame with 10 rows and 6 columns
               baseMean log2FoldChange     lfcSE      stat    pvalue      padj
              <numeric>      <numeric> <numeric> <numeric> <numeric> <numeric>
(TAACCC)n_1    0.195381     -0.4404094  3.084575 -0.142778  0.886466        NA
TAR1_2         0.699543      0.5526920  1.549296  0.356738  0.721288        NA
L1MC_3         0.535007      0.6424273  1.630292  0.394057  0.693539        NA
MER5B_4        0.518370      0.5945308  1.762943  0.337238  0.735938        NA
MIR3_5         1.380022     -0.7457321  1.081602 -0.689470  0.490527        NA
(TGCTCC)n_6    6.346198      0.0845385  0.546973  0.154557  0.877170  0.994814
Charlie15a_7   7.843825     -0.3306311  0.577770 -0.572253  0.567150  0.965960
(TGG)n_8       4.952008      0.3883758  0.633664  0.612905  0.539939  0.961462
L2a_9         21.516790     -0.3943097  0.

In [44]:
1

[1] 1

In [46]:
summary(res)


out of 4194671 with nonzero total read count
adjusted p-value < 0.1
LFC > 0 (up)       : 7369, 0.18%
LFC < 0 (down)     : 18805, 0.45%
outliers [1]       : 0, 0%
low counts [2]     : 2164159, 52%
(mean count < 2)
[1] see 'cooksCutoff' argument of ?results
[2] see 'independentFiltering' argument of ?results



In [47]:
# Convert the results to a data frame
res_df <- as.data.frame(res)

# Add the repeat GeneIDs to the results data frame
res_df$GeneID <- rownames(res_df)

# Filter for repeats that are significantly upregulated in TDP-43 negative samples
# A positive log2FoldChange means higher counts in the negative group.
# We'll use a significance threshold (adjusted p-value) of 0.05.
upregulated_repeats <- subset(res_df, log2FoldChange > 0 & padj < 0.05)

# Merge with your repeats_saf annotation to get genomic locations
# The 'repeats_saf' data frame should be the one you created initially.
merged_results <- merge(upregulated_repeats, repeats_saf, by = "GeneID")

# View the final table with locations
print(head(merged_results))

        GeneID  baseMean log2FoldChange     lfcSE     stat       pvalue
1 (A)n_2335023 49.274978       2.309329 0.4131560 5.589484 2.277448e-08
2 (A)n_2335031 57.145139       3.102640 0.4164526 7.450162 9.322543e-14
3 (A)n_2335212 77.387341       2.965991 0.3798939 7.807418 5.837126e-15
4 (A)n_2335296  4.738902       4.042215 0.9716044 4.160351 3.177587e-05
5 (A)n_2969004 41.325597       2.043413 0.5604351 3.646118 2.662316e-04
6 (A)n_3107262 37.577393       1.189326 0.3058248 3.888913 1.006942e-04
          padj   Chr    Start      End Strand
1 4.733186e-05  chr2  7058339  7058356      +
2 1.263776e-09  chr2  7062498  7062531      +
3 1.059761e-10  chr2  7157791  7157812      +
4 1.030174e-02  chr2  7208871  7208890      +
5 4.023491e-02 chr22 22261209 22261236      +
6 2.184737e-02  chr3 37881022 37881049      +


In [48]:
dim(merged_results)

[1] 4340   11

In [49]:
summary(merged_results)

    GeneID             baseMean        log2FoldChange       lfcSE       
 Length:4340        Min.   :   2.121   Min.   : 0.669   Min.   :0.1773  
 Class :character   1st Qu.:   9.029   1st Qu.: 1.674   1st Qu.:0.3960  
 Mode  :character   Median :  21.681   Median : 2.592   Median :0.5631  
                    Mean   :  62.329   Mean   : 2.801   Mean   :0.6370  
                    3rd Qu.:  59.035   3rd Qu.: 3.708   3rd Qu.:0.8317  
                    Max.   :3937.536   Max.   :22.751   Max.   :2.9881  
      stat            pvalue               padj              Chr           
 Min.   : 3.552   Min.   :0.000e+00   Min.   :0.000000   Length:4340       
 1st Qu.: 3.746   1st Qu.:2.229e-06   1st Qu.:0.001599   Class :character  
 Median : 4.043   Median :5.282e-05   Median :0.014373   Mode  :character  
 Mean   : 4.459   Mean   :1.020e-04   Mean   :0.017760                     
 3rd Qu.: 4.731   3rd Qu.:1.796e-04   3rd Qu.:0.031459                     
 Max.   :13.944   Max.   :3.829e-

In [51]:
head(merged_results,n=10)

,GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Chr,Start,End,Strand
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>
1,(A)n_2335023,49.274978,2.3093289,0.4131560,5.589484,2.277448e-08,4.733186e-05,chr2,7058339,7058356,+
2,(A)n_2335031,57.145139,3.1026397,0.4164526,7.450162,9.322543e-14,1.263776e-09,chr2,7062498,7062531,+
3,(A)n_2335212,77.387341,2.9659908,0.3798939,7.807418,5.837126e-15,1.059761e-10,chr2,7157791,7157812,+
4,(A)n_2335296,4.738902,4.0422155,0.9716044,4.160351,3.177587e-05,1.030174e-02,chr2,7208871,7208890,+
5,(A)n_2969004,41.325597,2.0434126,0.5604351,3.646118,2.662316e-04,4.023491e-02,chr22,22261209,22261236,+
6,(A)n_3107262,37.577393,1.1893260,0.3058248,3.888913,1.006942e-04,2.184737e-02,chr3,37881022,37881049,+
7,(A)n_3143310,10.423994,2.1464593,0.6025145,3.562502,3.673367e-04,4.877534e-02,chr3,56169809,56169827,+
8,(A)n_3966700,32.632315,1.5584793,0.4222902,3.690541,2.237772e-04,3.608052e-02,chr5,144299899,144299935,+
9,(AAAAG)n_2321107,7.262102,6.3441217,1.0555069,6.010498,1.849547e-09,6.417928e-06,chr19,57043173,57043215,+


In [52]:
tail(merged_results,n=10)

,GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Chr,Start,End,Strand
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>
4331,X4b_DNA_1260558,2.855308,4.115413,1.0269059,4.007585,6.134285e-05,1.586767e-02,chr13,44112146,44112222,-
4332,X4b_DNA_5165373,19.750399,4.280900,0.7308218,5.857652,4.694585e-09,1.349182e-05,chrX,18085721,18086137,-
4333,X5a_DNA_235417,7.798214,4.919386,1.2699872,3.873571,1.072520e-04,2.275469e-02,chr1,112087526,112087578,-
4334,X5a_DNA_3107329,77.853477,1.449357,0.3435568,4.218683,2.457339e-05,8.624098e-03,chr3,37918660,37918774,-
4335,X6a_DNA_4325305,17.128493,1.889352,0.5252189,3.597266,3.215796e-04,4.511255e-02,chr6,157576677,157576876,-
4336,X6b_DNA_4076458,229.565923,1.411731,0.2187168,6.454605,1.085012e-10,6.011678e-07,chr6,16608755,16608888,-
4337,X7A_LINE_2991764,35.885513,1.918128,0.4660126,4.116044,3.854308e-05,1.170814e-02,chr22,31898311,31898442,+
4338,X7B_LINE_5165014,5.356069,5.035311,0.9623378,5.232373,1.673473e-07,2.262549e-04,chrX,17920439,17920546,-
4339,X9_LINE_1349929,31.118209,1.431897,0.3439187,4.163475,3.134404e-05,1.020096e-02,chr13,96546578,96546637,+


In [53]:
head(repeatmasker_raw)

sw_score,perc_div,perc_del,perc_ins,chromosome,start,end,left,strand,rep_name,class_family,rep_start,rep_end,rep_left,id,dummy
<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>
463,1.3,0.6,1.7,chr1,10001,10468,(248945954),+,(TAACCC)n,Simple_repeat,1,463,(0),1,NA
3770,13.3,6.3,3.5,chr1,10469,11447,(248944975),C,TAR1,Satellite/telo,(399),1006,1,2,NA
535,20.4,18.1,2.5,chr1,11505,11675,(248944747),C,L1MC,LINE/L1,(2299),5648,5452,3,NA
263,29.4,1.9,1.0,chr1,11678,11780,(248944642),C,MER5B,DNA/hAT-Charlie,(74),104,1,4,NA
309,23.0,3.8,0.0,chr1,15265,15355,(248941067),C,MIR3,SINE/MIR,(119),143,49,5,NA
18,23.2,0.0,2.0,chr1,15798,15849,(248940573),+,(TGCTCC)n,Simple_repeat,1,51,(0),6,NA


In [54]:
repeats_saf2 <- repeatmasker_raw%>%
  transmute(
    GeneID = paste(rep_name, row_number(), sep = "_"),
    Chr = chromosome,
    Class_family = class_family,
    Start = start,
    End = end,
    Strand = case_when(
      strand == "+" ~ "+",
      strand == "C" ~ "-",
      TRUE ~ "*"
    )
  )

In [55]:
head(repeats_saf2)

GeneID,Chr,Class_family,Start,End,Strand
<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
(TAACCC)n_1,chr1,Simple_repeat,10001,10468,+
TAR1_2,chr1,Satellite/telo,10469,11447,-
L1MC_3,chr1,LINE/L1,11505,11675,-
MER5B_4,chr1,DNA/hAT-Charlie,11678,11780,-
MIR3_5,chr1,SINE/MIR,15265,15355,-
(TGCTCC)n_6,chr1,Simple_repeat,15798,15849,+


In [57]:
dim(repeats_saf2)

[1] 5448004       6

In [58]:
length(unique(repeats_saf2$GeneID))

[1] 5448004

In [56]:
head(merged_results,n=10)

,GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Chr,Start,End,Strand
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>
1,(A)n_2335023,49.274978,2.3093289,0.4131560,5.589484,2.277448e-08,4.733186e-05,chr2,7058339,7058356,+
2,(A)n_2335031,57.145139,3.1026397,0.4164526,7.450162,9.322543e-14,1.263776e-09,chr2,7062498,7062531,+
3,(A)n_2335212,77.387341,2.9659908,0.3798939,7.807418,5.837126e-15,1.059761e-10,chr2,7157791,7157812,+
4,(A)n_2335296,4.738902,4.0422155,0.9716044,4.160351,3.177587e-05,1.030174e-02,chr2,7208871,7208890,+
5,(A)n_2969004,41.325597,2.0434126,0.5604351,3.646118,2.662316e-04,4.023491e-02,chr22,22261209,22261236,+
6,(A)n_3107262,37.577393,1.1893260,0.3058248,3.888913,1.006942e-04,2.184737e-02,chr3,37881022,37881049,+
7,(A)n_3143310,10.423994,2.1464593,0.6025145,3.562502,3.673367e-04,4.877534e-02,chr3,56169809,56169827,+
8,(A)n_3966700,32.632315,1.5584793,0.4222902,3.690541,2.237772e-04,3.608052e-02,chr5,144299899,144299935,+
9,(AAAAG)n_2321107,7.262102,6.3441217,1.0555069,6.010498,1.849547e-09,6.417928e-06,chr19,57043173,57043215,+


In [59]:
repeats_saf2_uni <- repeats_saf2 %>% 
  distinct(GeneID, .keep_all = TRUE)

In [60]:
merged_with_ann <- merged_results %>% 
  left_join(
    repeats_saf2_uni %>% 
      select(GeneID, Chr, Start, End, Strand, Class_family),
    by = "GeneID",
    relationship = "many-to-one"   # <- will warn you if GeneID isn't unique
  )

In [61]:
glimpse(merged_with_ann)

Rows: 4,340
Columns: 16
$ GeneID         <chr> "(A)n_2335023", "(A)n_2335031", "(A)n_2335212", "(A)n_2…
$ baseMean       <dbl> 49.274978, 57.145139, 77.387341, 4.738902, 41.325597, 3…
$ log2FoldChange <dbl> 2.3093289, 3.1026397, 2.9659908, 4.0422155, 2.0434126, …
$ lfcSE          <dbl> 0.4131560, 0.4164526, 0.3798939, 0.9716044, 0.5604351, …
$ stat           <dbl> 5.589484, 7.450162, 7.807418, 4.160351, 3.646118, 3.888…
$ pvalue         <dbl> 2.277448e-08, 9.322543e-14, 5.837126e-15, 3.177587e-05,…
$ padj           <dbl> 4.733186e-05, 1.263776e-09, 1.059761e-10, 1.030174e-02,…
$ Chr.x          <chr> "chr2", "chr2", "chr2", "chr2", "chr22", "chr3", "chr3"…
$ Start.x        <dbl> 7058339, 7062498, 7157791, 7208871, 22261209, 37881022,…
$ End.x          <dbl> 7058356, 7062531, 7157812, 7208890, 22261236, 37881049,…
$ Strand.x       <chr> "+", "+", "+", "+", "+", "+", "+", "+", "+", "+", "+", …
$ Chr.y          <chr> "chr2", "chr2", "chr2", "chr2", "chr22", "chr3", "chr3"…
$ Start.y       

In [62]:
dim(merged_with_ann)

[1] 4340   16

In [63]:
head(merged_with_ann)

,GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Chr.x,Start.x,End.x,Strand.x,Chr.y,Start.y,End.y,Strand.y,Class_family
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,(A)n_2335023,49.274978,2.309329,0.4131560,5.589484,2.277448e-08,4.733186e-05,chr2,7058339,7058356,+,chr2,7058339,7058356,+,Simple_repeat
2,(A)n_2335031,57.145139,3.102640,0.4164526,7.450162,9.322543e-14,1.263776e-09,chr2,7062498,7062531,+,chr2,7062498,7062531,+,Simple_repeat
3,(A)n_2335212,77.387341,2.965991,0.3798939,7.807418,5.837126e-15,1.059761e-10,chr2,7157791,7157812,+,chr2,7157791,7157812,+,Simple_repeat
4,(A)n_2335296,4.738902,4.042215,0.9716044,4.160351,3.177587e-05,1.030174e-02,chr2,7208871,7208890,+,chr2,7208871,7208890,+,Simple_repeat
5,(A)n_2969004,41.325597,2.043413,0.5604351,3.646118,2.662316e-04,4.023491e-02,chr22,22261209,22261236,+,chr22,22261209,22261236,+,Simple_repeat
6,(A)n_3107262,37.577393,1.189326,0.3058248,3.888913,1.006942e-04,2.184737e-02,chr3,37881022,37881049,+,chr3,37881022,37881049,+,Simple_repeat


In [65]:
unique(merged_with_ann$Class_family)

[1] "Simple_repeat"     "rRNA"              "RNA"              
 [4] "Low_complexity"    "SINE/Alu"          "LINE/L2"          
 [7] "SINE/5S-Deu-L2"    "DNA/hAT-Tip100"    "scRNA"            
[10] "DNA/hAT-Blackjack" "DNA/hAT-Charlie"   "LINE/CR1"         
[13] "LTR/ERV1"          "LTR/ERVL"          "DNA"              
[16] "DNA/hAT-Ac"        "LTR/ERVL?"         "DNA/hAT-Tip100?"  
[19] "LTR?"              "Unknown"           "LINE/L1"          
[22] "DNA/hAT-Tag1"      "LTR/ERVK"          "DNA/TcMar-Mariner"
[25] "DNA/TcMar-Tc2"     "LINE/RTE-X"        "SINE/tRNA"        
[28] "LTR/ERV1?"         "LTR/Gypsy"         "LTR"              
[31] "LTR/Gypsy?"        "DNA/hAT"           "DNA/TcMar-Tigger" 
[34] "LINE/RTE-BovB"     "SINE/tRNA-RTE"     "DNA?"             
[37] "DNA/PiggyBac"      "SINE/MIR"          "LTR/ERVL-MaLR"    
[40] "LINE/Penelope"     "DNA/MULE-MuDR"     "Retroposon/SVA"   
[43] "DNA/hAT?"          "DNA/TcMar"

In [66]:
write_csv(
  merged_with_ann,
  here("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam", "data", "repeat_families_in_excess.csv")
)

In [67]:
merged_with_ann[merged_with_ann$GeneID=='MIR3_1729538',]

,GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Chr.x,Start.x,End.x,Strand.x,Chr.y,Start.y,End.y,Strand.y,Class_family
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>
3420,MIR3_1729538,16.22163,22.75068,2.98811,7.613736,2.66285e-14,4.10204e-10,chr16,10592026,10592079,-,chr16,10592026,10592079,-,SINE/MIR


In [68]:
merged_with_ann[merged_with_ann$log2FoldChange>=7,]

,GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Chr.x,Start.x,End.x,Strand.x,Chr.y,Start.y,End.y,Strand.y,Class_family
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>
76,(ATAG)n_2321098,14.37944,7.333977,1.1074189,6.622586,3.529680e-11,2.315265e-07,chr19,57040543,57040849,+,chr19,57040543,57040849,+,Simple_repeat
101,(CA)n_235340,15.83409,7.468087,1.0637670,7.020416,2.212083e-12,2.111781e-08,chr1,112042116,112042146,+,chr1,112042116,112042146,+,Simple_repeat
120,(CACAGT)n_1880749,14.64334,7.359933,1.1372838,6.471501,9.703420e-11,5.496136e-07,chr16,89023050,89023468,+,chr16,89023050,89023468,+,Simple_repeat
295,(TGGA)n_2321109,52.12763,8.363042,0.9965183,8.392262,4.768783e-17,1.539197e-12,chr19,57044012,57044703,+,chr19,57044012,57044703,+,Simple_repeat
361,AluJb_2321080,21.65905,7.505752,1.7240721,4.353502,1.339796e-05,5.688805e-03,chr19,57033248,57033559,+,chr19,57033248,57033559,+,SINE/Alu
611,AluSg_5990,24.93398,7.291266,1.3824418,5.274194,1.333411e-07,1.876391e-04,chr1,4227045,4227335,-,chr1,4227045,4227335,-,SINE/Alu
1952,L1PA15_2321144,17.49879,7.199227,1.1548341,6.233992,4.546954e-10,2.023166e-06,chr19,57056235,57056694,+,chr19,57056235,57056694,+,LINE/L1
2174,L2a_2321115,36.30093,7.838917,1.0356954,7.568747,3.768404e-14,5.593246e-10,chr19,57047780,57048262,+,chr19,57047780,57048262,+,LINE/L2
2432,L2c_2321108,22.85158,8.000272,1.0243366,7.810198,5.709800e-15,1.055493e-10,chr19,57043283,57043896,+,chr19,57043283,57043896,+,LINE/L2


In [69]:
df_sorted <- merged_with_ann[order(merged_with_ann$log2FoldChange, decreasing = TRUE), ]

In [70]:
head(df_sorted,n=10)

,GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Chr.x,Start.x,End.x,Strand.x,Chr.y,Start.y,End.y,Strand.y,Class_family
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>
3420,MIR3_1729538,16.22163,22.750676,2.9881095,7.613736,2.662850e-14,4.102040e-10,chr16,10592026,10592079,-,chr16,10592026,10592079,-,SINE/MIR
295,(TGGA)n_2321109,52.12763,8.363042,0.9965183,8.392262,4.768783e-17,1.539197e-12,chr19,57044012,57044703,+,chr19,57044012,57044703,+,Simple_repeat
2432,L2c_2321108,22.85158,8.000272,1.0243366,7.810198,5.709800e-15,1.055493e-10,chr19,57043283,57043896,+,chr19,57043283,57043896,+,LINE/L2
2174,L2a_2321115,36.30093,7.838917,1.0356954,7.568747,3.768404e-14,5.593246e-10,chr19,57047780,57048262,+,chr19,57047780,57048262,+,LINE/L2
3826,MLT1B_2321143,16.83717,7.559768,1.9187147,3.940017,8.147590e-05,1.913324e-02,chr19,57055426,57055816,-,chr19,57055426,57055816,-,LTR/ERVL-MaLR
361,AluJb_2321080,21.65905,7.505752,1.7240721,4.353502,1.339796e-05,5.688805e-03,chr19,57033248,57033559,+,chr19,57033248,57033559,+,SINE/Alu
101,(CA)n_235340,15.83409,7.468087,1.0637670,7.020416,2.212083e-12,2.111781e-08,chr1,112042116,112042146,+,chr1,112042116,112042146,+,Simple_repeat
120,(CACAGT)n_1880749,14.64334,7.359933,1.1372838,6.471501,9.703420e-11,5.496136e-07,chr16,89023050,89023468,+,chr16,89023050,89023468,+,Simple_repeat
76,(ATAG)n_2321098,14.37944,7.333977,1.1074189,6.622586,3.529680e-11,2.315265e-07,chr19,57040543,57040849,+,chr19,57040543,57040849,+,Simple_repeat


In [72]:
dim(df_sorted[df_sorted$Class_family=='Simple_repeat',])

[1] 323  16

In [73]:
dim(df_sorted[df_sorted$Class_family=='SINE/MIR',])

[1] 503  16

In [75]:
# SINE/Alu
dim(df_sorted[df_sorted$Class_family=='SINE/Alu',])

[1] 656  16

In [76]:
# SINE/Alu
dim(df_sorted[df_sorted$Class_family=='LINE/L2',])

[1] 525  16

In [77]:
# SINE/Alu
dim(df_sorted[df_sorted$Class_family=='LINE/L1',])

[1] 893  16

# additional tests - Unique only mapping of reads

In [23]:
# Extract the count matrix from your featureCounts results
count_matrix <- fc_results$counts

In [24]:
dim(count_matrix)

[1] 5448004      14

In [25]:
head(count_matrix)

,SRR8571937.sorted.bam,SRR8571938.sorted.bam,SRR8571939.sorted.bam,SRR8571940.sorted.bam,SRR8571941.sorted.bam,SRR8571942.sorted.bam,SRR8571944.sorted.bam,SRR8571945.sorted.bam,SRR8571947.sorted.bam,SRR8571948.sorted.bam,SRR8571949.sorted.bam,SRR8571950.sorted.bam,SRR8571951.sorted.bam,SRR8571952.sorted.bam
(TAACCC)n_1,0,4,0,1,0,0,1,0,0,0,4,0,4,0
TAR1_2,0,0,0,0,2,2,2,3,5,6,0,0,0,2
L1MC_3,6,3,2,1,1,0,0,2,3,9,3,6,1,4
MER5B_4,9,3,3,0,2,0,0,6,4,8,4,5,0,1
MIR3_5,3,6,14,7,2,1,13,3,5,4,12,10,0,0
(TGCTCC)n_6,12,9,24,23,10,18,20,32,23,18,28,13,10,2


In [26]:
sample_info <- data.frame(
  sample_name = colnames(count_matrix),
  condition = factor(c("TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative","TDP43_positive","TDP43_negative")) # Adjust this to your actual sample conditions
)
rownames(sample_info) <- colnames(count_matrix)

In [27]:
# Create the DESeqDataSet object
dds <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = sample_info,
  design = ~ condition
)

In [ ]:
# Run the DESeq2 analysis
dds <- DESeq(dds)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [29]:
1

[1] 1

In [30]:
qsave(
  dds,
  file.path("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam", "data", "tdp-43_path_de-only_unique_mappings.qs")
)

In [31]:
res <- results(dds, contrast = c("condition", "TDP43_negative", "TDP43_positive"))

In [32]:
head(res,n=10)

log2 fold change (MLE): condition TDP43_negative vs TDP43_positive 
Wald test p-value: condition TDP43_negative vs TDP43_positive 
DataFrame with 10 rows and 6 columns
               baseMean log2FoldChange     lfcSE       stat    pvalue      padj
              <numeric>      <numeric> <numeric>  <numeric> <numeric> <numeric>
(TAACCC)n_1    0.995715     -0.8899925  1.766687 -0.5037634  0.614428        NA
TAR1_2         1.489474      0.4539081  1.345312  0.3373999  0.735815        NA
L1MC_3         2.734474      0.5640810  0.848601  0.6647188  0.506230  0.949158
MER5B_4        2.980793     -0.0273228  0.955334 -0.0286003  0.977183  0.998939
MIR3_5         5.159582     -0.6852746  0.755026 -0.9076172  0.364081  0.906916
(TGCTCC)n_6   16.124224     -0.1688535  0.454724 -0.3713318  0.710390  0.980075
Charlie15a_7  11.076430     -0.4990029  0.579304 -0.8613828  0.389027  0.916081
(TGG)n_8      18.157096      0.5437366  0.510616  1.0648632  0.286938  0.870135
L2a_9         77.557866     -0.3

In [33]:
head(results(dds, contrast = c("condition",  "TDP43_positive", "TDP43_negative")),n=10)

log2 fold change (MLE): condition TDP43_positive vs TDP43_negative 
Wald test p-value: condition TDP43 positive vs TDP43 negative 
DataFrame with 10 rows and 6 columns
               baseMean log2FoldChange     lfcSE       stat    pvalue      padj
              <numeric>      <numeric> <numeric>  <numeric> <numeric> <numeric>
(TAACCC)n_1    0.995715      0.8899925  1.766687  0.5037634  0.614428        NA
TAR1_2         1.489474     -0.4539081  1.345312 -0.3373999  0.735815        NA
L1MC_3         2.734474     -0.5640810  0.848601 -0.6647188  0.506230  0.949158
MER5B_4        2.980793      0.0273228  0.955334  0.0286003  0.977183  0.998939
MIR3_5         5.159582      0.6852746  0.755026  0.9076172  0.364081  0.906916
(TGCTCC)n_6   16.124224      0.1688535  0.454724  0.3713318  0.710390  0.980075
Charlie15a_7  11.076430      0.4990029  0.579304  0.8613828  0.389027  0.916081
(TGG)n_8      18.157096     -0.5437366  0.510616 -1.0648632  0.286938  0.870135
L2a_9         77.557866      0.3

In [34]:
summary(res)


out of 4514534 with nonzero total read count
adjusted p-value < 0.1
LFC > 0 (up)       : 8242, 0.18%
LFC < 0 (down)     : 21441, 0.47%
outliers [1]       : 0, 0%
low counts [2]     : 2253717, 50%
(mean count < 2)
[1] see 'cooksCutoff' argument of ?results
[2] see 'independentFiltering' argument of ?results



In [35]:
# Convert the results to a data frame
res_df <- as.data.frame(res)

# Add the repeat GeneIDs to the results data frame
res_df$GeneID <- rownames(res_df)

# Filter for repeats that are significantly upregulated in TDP-43 negative samples
# A positive log2FoldChange means higher counts in the negative group.
# We'll use a significance threshold (adjusted p-value) of 0.05.
upregulated_repeats <- subset(res_df, log2FoldChange > 0 & padj < 0.05)

# Merge with your repeats_saf annotation to get genomic locations
# The 'repeats_saf' data frame should be the one you created initially.
merged_results <- merge(upregulated_repeats, repeats_saf, by = "GeneID")

# View the final table with locations
print(head(merged_results))

            GeneID  baseMean log2FoldChange     lfcSE     stat       pvalue
1     (A)n_2335023 71.930045       2.283800 0.4437071 5.147089 2.645595e-07
2     (A)n_2335031 77.425629       3.020220 0.3990349 7.568812 3.766534e-14
3     (A)n_2335212 92.296721       2.902504 0.3887683 7.465897 8.273386e-14
4     (A)n_2335296  4.732344       4.046540 0.9761940 4.145221 3.394857e-05
5     (A)n_2969004 75.615904       2.177367 0.5605156 3.884579 1.025074e-04
6 (AAAAC)n_3653885 10.020593       4.099816 1.0440725 3.926754 8.609989e-05
          padj   Chr     Start       End Strand
1 3.370053e-04  chr2   7058339   7058356      +
2 6.223321e-10  chr2   7062498   7062531      +
3 1.248512e-09  chr2   7157791   7157812      +
4 1.073871e-02  chr2   7208871   7208890      +
5 2.207633e-02 chr22  22261209  22261236      +
6 1.979044e-02  chr4 147860436 147860474      +


In [36]:
dim(merged_results)

[1] 4773   11

In [37]:
repeats_saf2 <- repeatmasker_raw%>%
  transmute(
    GeneID = paste(rep_name, row_number(), sep = "_"),
    Chr = chromosome,
    Class_family = class_family,
    Start = start,
    End = end,
    Strand = case_when(
      strand == "+" ~ "+",
      strand == "C" ~ "-",
      TRUE ~ "*"
    )
  )

In [38]:
repeats_saf2_uni <- repeats_saf2 %>% 
  distinct(GeneID, .keep_all = TRUE)

In [39]:
merged_with_ann <- merged_results %>% 
  left_join(
    repeats_saf2_uni %>% 
      select(GeneID, Chr, Start, End, Strand, Class_family),
    by = "GeneID",
    relationship = "many-to-one"   # <- will warn you if GeneID isn't unique
  )

In [40]:
dim(merged_with_ann)

[1] 4773   16

In [41]:
write_csv(
  merged_with_ann,
  here("dsrna_STAR-fractional_AND_secondary-noMETA_11_fam", "data", "repeat_families_in_excess-only_unique_mappings.csv")
)

In [42]:
merged_with_ann[merged_with_ann['log2FoldChange']>=7,]

,GeneID,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Chr.x,Start.x,End.x,Strand.x,Chr.y,Start.y,End.y,Strand.y,Class_family
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>
134,(CA)n_235340,19.97369,7.811417,1.0686561,7.309570,2.679988e-13,3.632596e-09,chr1,112042116,112042146,+,chr1,112042116,112042146,+,Simple_repeat
359,(TGGA)n_2321109,55.18244,8.034416,0.9952865,8.072466,6.889227e-16,1.856488e-11,chr19,57044012,57044703,+,chr19,57044012,57044703,+,Simple_repeat
1235,ERVL-E-int_4340231,42.13035,7.267967,0.8230616,8.830405,1.042976e-18,6.380777e-14,chr6,166396038,166396307,+,chr6,166396038,166396307,+,LTR/ERVL
2069,L1ME4b_2321097,11.47702,7.016761,1.1989405,5.852468,4.843313e-09,1.459834e-05,chr19,57040436,57040527,+,chr19,57040436,57040527,+,LINE/L1
2210,L1PA15_2321144,17.58535,7.213923,1.1603221,6.217173,5.061927e-10,2.255553e-06,chr19,57056235,57056694,+,chr19,57056235,57056694,+,LINE/L1
2445,L2a_2321115,43.58122,7.701945,1.0173317,7.570731,3.711291e-14,6.177132e-10,chr19,57047780,57048262,+,chr19,57047780,57048262,+,LINE/L2
2737,L2c_2321108,26.19537,7.789816,1.0088007,7.721858,1.146457e-14,2.199260e-10,chr19,57043283,57043896,+,chr19,57043283,57043896,+,LINE/L2
2978,LTR103b_Mam_235341,16.22974,7.092553,1.0470751,6.773682,1.255459e-11,1.048658e-07,chr1,112042181,112042356,-,chr1,112042181,112042356,-,LTR/ERV1?
3802,MIR3_1876258,11.10042,22.264576,2.9887836,7.449377,9.378206e-14,1.405865e-09,chr16,86490475,86490578,+,chr16,86490475,86490578,+,SINE/MIR
